# 04 – Feature Engineering

Cyclical day-of-year encoding from `date_of_record`.

- Adds `doy_sin`, `doy_cos`
- Drops redundant `month` / `season` text columns
- Keeps `station_name`, `state`, `district` as metadata (not model inputs yet)

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Resolve paths whether the notebook is run from notebooks/ or project root
BASE = Path("..").resolve()
if not (BASE / "data" / "processed" / "clean_dataset.csv").exists():
    BASE = Path(".").resolve()

CLEAN_PATH = BASE / "data" / "processed" / "clean_dataset.csv"
OUT_PATH = BASE / "data" / "processed" / "feature_engineered.csv"

# 1. Load cleaned dataset
df = pd.read_csv(CLEAN_PATH)
df["date_of_record"] = pd.to_datetime(df["date_of_record"])

print("Loaded:", df.shape)
df.head()

Loaded: (712785, 15)


,date_of_record,month,season,station_name,state,district,avg_temp,min_temp,max_temp,wind_speed,air_pressure,elevation,latitude,longitude,rainfall
0,2015-01-02,January,Winter,Agartala,TR,West Tripura,23.1,18.600000,30.3,4.4,1009.2,15,23.8833,91.25,0.0
1,2015-01-17,January,Winter,Agartala,TR,West Tripura,20.4,18.593333,27.4,4.4,1009.2,15,23.8833,91.25,0.3
2,2015-01-18,January,Winter,Agartala,TR,West Tripura,15.8,18.586667,17.0,4.4,1009.2,15,23.8833,91.25,2.0
3,2015-01-19,January,Winter,Agartala,TR,West Tripura,15.2,18.580000,17.7,4.4,1009.2,15,23.8833,91.25,0.0
4,2015-02-15,February,Winter,Agartala,TR,West Tripura,19.7,18.573333,27.0,4.4,1009.2,15,23.8833,91.25,2.0


In [2]:
# 2. Sanity check before we touch anything
n_rows_before = len(df)
assert df["date_of_record"].isna().sum() == 0, "Found unparseable dates — stop and investigate"
print(f"Rows: {n_rows_before:,}")
print("Columns:", df.columns.tolist())

Rows: 712,785
Columns: ['date_of_record', 'month', 'season', 'station_name', 'state', 'district', 'avg_temp', 'min_temp', 'max_temp', 'wind_speed', 'air_pressure', 'elevation', 'latitude', 'longitude', 'rainfall']


In [3]:
# 3. Day-of-year cyclical encoding
# use 366 so leap years don't distort the phase
day_of_year = df["date_of_record"].dt.dayofyear
df["doy_sin"] = np.sin(2 * np.pi * day_of_year / 366)
df["doy_cos"] = np.cos(2 * np.pi * day_of_year / 366)

# 4. Drop redundant categorical calendar columns
df = df.drop(columns=["month", "season"])

# 5. state / district / station_name kept as metadata (not encoded here)

In [4]:
# 6. Verification before saving
assert len(df) == n_rows_before, "Row count changed - something went wrong"
assert df["doy_sin"].between(-1, 1).all(), "doy_sin out of expected range"
assert df["doy_cos"].between(-1, 1).all(), "doy_cos out of expected range"
assert df.isna().sum().sum() == 0, "NaNs introduced during feature engineering"

# 7. Save
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUT_PATH, index=False)
print(f"Saved {len(df):,} rows, {df.shape[1]} columns -> {OUT_PATH}")
print(df[["date_of_record", "doy_sin", "doy_cos"]].head())

Saved 712,785 rows, 15 columns -> D:\project\Research Project\RainfallPrediction\data\processed\feature_engineered.csv
  date_of_record   doy_sin   doy_cos
0     2015-01-02  0.034328  0.999411
1     2015-01-17  0.287717  0.957716
2     2015-01-18  0.304115  0.952635
3     2015-01-19  0.320423  0.947274
4     2015-02-15  0.710135  0.704066


## Post-save verification

In [5]:
check = pd.read_csv(OUT_PATH)
print(check.shape)
print(check.columns.tolist())
print(check[["doy_sin", "doy_cos"]].describe())

(712785, 15)
['date_of_record', 'station_name', 'state', 'district', 'avg_temp', 'min_temp', 'max_temp', 'wind_speed', 'air_pressure', 'elevation', 'latitude', 'longitude', 'rainfall', 'doy_sin', 'doy_cos']
             doy_sin        doy_cos
count  712785.000000  712785.000000
mean       -0.032170      -0.044503
std         0.697787       0.714199
min        -0.999963      -1.000000
25%        -0.722117      -0.762354
50%        -0.051479      -0.094279
75%         0.647161       0.679273
max         0.999963       1.000000
